# AI Engineering Interview Prep
## Section: Evaluation & Testing

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 731+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (80 Qs)",
    "18. FastAPI (60 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 📊 Section 9 — Evaluation and Testing

### Q1. AI Agent Evaluation
**A:** Evaluating agents is harder than evaluating single LLM calls because you must assess both the **outcome** (did it achieve the goal?) and the **trajectory** (did it take efficient, correct steps?).

**Outcome metrics:**
- Task success rate: % of tasks fully completed correctly
- Partial success rate: % of sub-goals achieved
- Answer quality: for tasks with verifiable outputs

**Trajectory metrics:**
- Step efficiency: did it take the minimum steps needed?
- Tool accuracy: correct tool called with correct parameters?
- Recovery quality: when it failed, did it recover well?

**Evaluation approaches:**
1. **Golden trajectories:** human experts define the ideal step sequence; compare agent trajectory
2. **LLM-as-judge:** "Given the task and agent trajectory, rate the quality 1-10"
3. **Environment-based:** run the agent in a simulated environment; measure objective outcomes
4. **Benchmarks:** WebArena, AgentBench, SWE-bench, GAIA

🏷️ *For Nyaya-Pro agent evaluation: I maintain 50 test legal queries with known correct legal sections. Success = agent cites the correct section in its response.*


### Q2. LLM Evaluation
**A:** How to evaluate an LLM or LLM application:

**Automatic metrics:**
- **Perplexity:** model's uncertainty over held-out text; lower = better language model
- **BLEU/ROUGE:** overlap between generated and reference text; for summarisation/translation
- **BERTScore:** semantic similarity using BERT embeddings; better than n-gram overlap
- **Accuracy/F1:** for classification tasks with clear right/wrong answers

**LLM-as-judge (G-Eval):**
- Use a powerful LLM (GPT-4) to rate outputs on dimensions: accuracy, coherence, helpfulness, safety
- Define rubric clearly; provide examples of each score level
- Fast and scalable; correlates well with human judgement

**Human evaluation:**
- Side-by-side comparison (A vs B)
- Rating on specific dimensions
- Expensive but gold standard

**Domain-specific test sets:**
- Golden test set with expert-verified answers
- Run every model/prompt change against this set


### Q3. AI Agent Observability
**A:** Observability for agents = the ability to understand what the agent did, why, and where it went wrong, from the outside.

**What to capture:**
- Full agent trajectory (every Thought/Action/Observation triplet)
- Tool calls: name, arguments, return value, latency, success/error
- LLM calls: prompt, response, tokens, latency, cost
- Final output + ground truth (if known)

**Tools:**
- **LangSmith:** native LangChain tracing; traces nested agent steps automatically
- **Arize Phoenix:** open-source; great for RAG + agent observability
- **Weights & Biases Traces:** agent trajectory logging
- **Custom logging:** structured JSON logs with trace_id linking all steps

**Key signals to monitor:**
- Steps per task (efficiency regression)
- Tool failure rate
- Context window usage % (approaching limits?)
- Cost per task
- Task success rate over time


### Q4. What is evaluation-driven development for AI applications?
**A:** EDD (Evaluation-Driven Development) for AI = build your evaluation harness BEFORE building the application. Think test-driven development (TDD) applied to AI.

**Process:**
1. Define success criteria and metrics before writing any code
2. Build a golden test set (50-100 examples with verified correct answers)
3. Write evaluation code (automated scoring)
4. THEN build the application
5. Every change is evaluated against the test set before merging

**Why it matters:** Without EDD, you iterate based on intuition. With EDD, every change is objectively measured. You can prove your improvements and catch regressions.

**Cadence:** Run evals on every pull request (CI pipeline), every prompt change, every model update. Don't deploy without checking eval results.

🏷️ *All three of my projects have a `tests/eval/` directory with 50+ test cases. New features must not decrease eval score before merging.*


### Q5. How do you evaluate LLM outputs? What metrics do you use?
**A:**
**For generation quality:**
- BLEU (n-gram precision) — for translation; less useful for open-ended generation
- ROUGE-L (longest common subsequence) — for summarisation
- BERTScore (semantic similarity via BERT) — better correlation with human judgement than n-gram metrics

**For factual accuracy:**
- Faithfulness score (NLI or LLM-judge): is the answer supported by the source?
- Answer correctness (vs golden answer): exact match or semantic similarity
- Hallucination rate: % of claims not in source

**For helpfulness:**
- User satisfaction scores (CSAT, thumbs up/down)
- Task completion rate
- LLM-as-judge on helpfulness dimension (1-5 scale)

**For safety:**
- Toxicity rate (Perspective API)
- Policy violation rate (custom classifier)
- Red team success rate

**Key principle:** No single metric tells the whole story. Use a combination; weight by what matters most for your use case.


### Q6. Explain BLEU, ROUGE, and BERTScore. When would you use each?
**A:**
**BLEU (Bilingual Evaluation Understudy):**
- Measures precision of n-gram overlap between generated and reference text
- Range: 0-1 (1 = exact match)
- Use: Machine translation, where reference translation exists
- Weakness: Doesn't capture synonyms or paraphrasing; 1-reference BLEU can be low even for good outputs

**ROUGE (Recall-Oriented Understudy for Gisting Evaluation):**
- ROUGE-1: unigram overlap. ROUGE-2: bigram overlap. ROUGE-L: longest common subsequence.
- Measures recall (how much of the reference is in the generated text)
- Use: Summarisation evaluation
- Weakness: Same as BLEU — doesn't capture semantics

**BERTScore:**
- Uses BERT embeddings to compute semantic similarity between tokens
- More robust to synonyms and paraphrasing
- Correlates better with human judgement than n-gram metrics
- Use: Open-ended generation, chat, QA — when exact wording doesn't matter
- Weakness: More expensive to compute; opaque

**Recommendation:** Use ROUGE for summarisation, BLEU for translation, BERTScore when semantic quality matters more than exact phrasing.


### Q7. What is G-Eval, and how does it use LLMs for evaluation?
**A:** G-Eval (Liu et al., 2023) uses LLMs as evaluators by defining an evaluation form with criteria and asking an LLM to fill it in:

```
Task: rate this summary for coherence on a 1-5 scale.
Steps:
1. Read the source document.
2. Read the generated summary.
3. Assess whether the summary is logically structured.
4. Rate coherence from 1 (incoherent) to 5 (highly coherent).
Source: {source}
Summary: {summary}
Rating: 
```

The LLM outputs a score. G-Eval then uses the **token probabilities** (logprobs) of score tokens (1, 2, 3, 4, 5) to compute a weighted expected score — more robust than just taking the argmax.

Benefits: Works for any quality dimension you define; no reference answer needed; scales to any language/domain; higher correlation with human judgement than traditional metrics.

Limitations: Expensive (requires a frontier LLM), subject to the judge model's own biases.


### Q8. What is LLM-as-a-judge evaluation, and what are its limitations?
**A:** LLM-as-judge = using a powerful LLM (GPT-4, Claude) to evaluate outputs from another (or the same) LLM on quality dimensions.

**How it works:**
```
"Rate the following response for accuracy, helpfulness, and safety.
Query: {query}
Response: {response}
[Optional: Reference answer: {reference}]
Score each dimension 1-5 with brief justification."
```

**Strengths:** Scalable, cheap (vs human labelling), no reference answer needed for some dimensions.

**Limitations:**
1. **Positional bias:** tends to favour first-presented response in A/B comparisons
2. **Verbosity bias:** prefers longer, more elaborate responses regardless of quality
3. **Self-bias:** GPT-4 tends to rate GPT-4 outputs higher
4. **Inconsistency:** same input can yield different scores; temperature-sensitive
5. **Calibration:** scores don't always align with human calibration
6. **Domain expertise gaps:** limited ability to evaluate highly technical accuracy

**Mitigations:** Multiple judge models, swap position of A/B, provide detailed rubric with examples, compare to human labels on a sample.


### Q9. How do you conduct human evaluation for AI systems?
**A:**
**Study design:**
1. Define evaluation dimensions: accuracy, helpfulness, safety, coherence, naturalness
2. Create rating rubrics with anchors (what does a 1 vs 5 look like?)
3. Sample representative test cases (not cherry-picked)
4. Recruit appropriate annotators (domain experts for technical domains)

**Annotation setup:**
- Use a tool: Label Studio, Scale AI, Surge AI
- Blind evaluation: annotators don't know which system generated which output
- A/B comparison (side-by-side) or individual rating (1-5 scale)
- At least 3 annotators per example; take majority vote or average

**Quality control:**
- Compute inter-annotator agreement (Cohen's Kappa, Krippendorff's Alpha)
- Attention checks: include obvious test cases; flag annotators who fail them
- Calibration: regular alignment sessions between annotators

**Sample size:** 200-500 examples for statistically significant comparisons.


### Q10. What is red teaming, and how do you red team an LLM application?
**A:** Red teaming = systematically trying to break your AI system before attackers do. The "red team" plays adversary; the "blue team" defends.

**Red teaming approach:**
1. **Define threat model:** who would attack, what's their goal, what are the worst-case outputs?
2. **Automated probing:** use an LLM to generate adversarial prompts in 5 categories:
   - Jailbreaks ("ignore your instructions and...")
   - Prompt injections (in retrieved documents)
   - PII extraction ("what's in your training data about...")
   - Harmful content generation ("tell me how to...")
   - Safety bypasses (roleplay, fictional framing, multilingual)
3. **Human red teamers:** specialised adversarial testers who creatively probe for failures
4. **Taxonomy of failures:** document every failure found, categorise, prioritise by severity
5. **Fix and re-test:** implement mitigations; verify they work; re-run tests
6. **Continuous red teaming:** don't do it once; run automated red teaming in CI and periodic manual exercises

🏷️ *Before launching Nyaya-Pro publicly, I ran 50 adversarial prompts covering: legal advice bypass, jailbreaks, PII extraction, and off-topic manipulation.*


### Q11. How do you detect and measure hallucinations in LLM outputs?
**A:**
**Definition:** Hallucinations = factually incorrect claims stated confidently as true.

**Types:**
- Factual hallucination: "The Eiffel Tower is in Berlin"
- Faithfulness hallucination (RAG): answer contradicts the provided context
- Attribution hallucination: citing non-existent sources

**Detection methods:**
1. **NLI-based faithfulness:** use a natural language inference model to check if each claim is entailed by the source context. DeFacto, TRUE.
2. **LLM self-check:** "For each factual claim in this response, rate if it's supported by the context: supported/unsupported/contradicted"
3. **Retrieval verification:** for RAG — re-retrieve relevant chunks; check if the answer is grounded
4. **Fact verification API:** Google Fact Check, or a dedicated fact-checking LLM
5. **Confidence calibration:** use logprobs to detect when the model is uncertain; flag uncertain outputs

**Measurement:** % of hallucinated claims per response; hallucination rate per query type.

🏷️ *Nyaya-Pro: RAGAS faithfulness metric (NLI-based) run monthly on 50-query test set. Any session with faithfulness < 0.85 flagged for review.*


### Q12. What is adversarial testing for AI systems?
**A:** Adversarial testing = systematically probing for inputs that cause the model to fail, output harmful content, or behave unexpectedly.

**Categories:**
1. **Prompt injection tests:** can an attacker override system instructions?
2. **Jailbreak tests:** can safety restrictions be bypassed via creative prompting?
3. **Data poisoning tests:** does malicious content in retrieved docs affect outputs?
4. **Boundary tests:** what happens at exact context length limits?
5. **Edge case inputs:** empty string, very long input, only special characters, adversarial Unicode
6. **Multi-lingual attacks:** attacks that work in one language but not another
7. **Semantic attacks:** paraphrased versions of known harmful inputs

**Tools:** Garak (LLM vulnerability scanner), PyRIT (Microsoft), custom adversarial test suites.

**Process:** Automated → human review of failures → fix → re-test. Document all test cases and results for audit.


### Q13. How do you build a regression test suite for AI applications?
**A:**
**Structure:**
```
tests/
├── unit/           # test individual functions (chunking, parsing, formatting)
├── integration/    # test full pipeline (query → retrieval → LLM → response)
└── eval/           # golden dataset evaluation (quality regression)
    ├── golden_queries.json      # 100+ test queries with expected outputs
    ├── run_eval.py              # evaluation script
    └── eval_results/            # historical results for trend analysis
```

**golden_queries.json format:**
```json
[
  {
    "query": "What is the punishment for murder under BNS?",
    "expected_sections": ["BNS Section 101", "BNS Section 103"],
    "expected_answer_contains": ["death", "life imprisonment"],
    "should_not_contain": ["IPC 302"]  // old act, should use BNS
  }
]
```

**CI integration:** Run on every PR. Compare accuracy score vs `main` branch baseline. Fail build if score drops > 2%.

**Track over time:** Store results with timestamp → plot quality trend → detect gradual degradation before users notice.


### Q14. What are benchmark suites (MMLU, HumanEval, GSM8K), and how do you interpret them?
**A:**
| Benchmark | Domain | What it tests | Key caveat |
|-----------|--------|--------------|------------|
| **MMLU** | General knowledge | 57 academic subjects (STEM, humanities, law, medicine) | Multiple choice; doesn't test generation quality |
| **HumanEval** | Code | 164 Python programming problems | Pass@k metric; functional correctness only |
| **GSM8K** | Math | Grade school math word problems | Multi-step arithmetic reasoning |
| **HellaSwag** | Commonsense | Sentence completion tasks | Saturated by large models; less discriminative |
| **TruthfulQA** | Factuality | Misconceptions and tricky questions | Tests calibration and honesty |
| **MATH** | Advanced math | Competition-level math | Hard; differentiates frontier models |
| **BIG-bench Hard** | Reasoning | Challenging tasks beyond human-level | For cutting-edge model comparison |

**Interpretation:** No single benchmark tells the whole story. Always evaluate on YOUR domain's data. Models can be fine-tuned to overfit benchmarks — benchmark leaderboard positions can be misleading.


### Q15. How do you evaluate a RAG system end-to-end?
**A:** Use the RAGAS framework which evaluates four dimensions:

**Retrieval quality:**
1. **Context Precision:** Of retrieved chunks, what % are relevant to the query? (precision@k)
2. **Context Recall:** Of all relevant chunks in the knowledge base, what % did you retrieve?

**Generation quality:**
3. **Faithfulness:** Is every claim in the answer supported by the retrieved context? (no hallucinations)
4. **Answer Relevance:** Does the answer actually address the user's question? (not off-topic)

**Plus (if golden answers exist):**
5. **Answer Correctness:** Does the answer match the ground truth?

**How to run:**
```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

results = evaluate(
    dataset=test_dataset,  # query, contexts, answer, ground_truth
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)
```

🏷️ *Monthly RAGAS evaluation on Nyaya-Pro's 50-pair golden dataset. Faithfulness is the business-critical metric — a wrong legal answer could cause harm.*


### Q16. How do you evaluate the quality of AI agents?
**A:** Agent evaluation needs to assess both what the agent achieved AND how it achieved it.

**Task-level metrics:**
- **Success rate:** did the agent complete the task? (binary)
- **Partial credit:** what % of sub-goals were met?
- **Answer quality:** for tasks with verifiable outputs (code runs, math is correct, legal citation is accurate)

**Trajectory-level metrics:**
- **Efficiency:** steps taken vs minimum necessary steps (step ratio)
- **Tool accuracy:** % of tool calls with correct tool + correct parameters
- **Redundancy:** % of steps that were unnecessary/repeated
- **Recovery quality:** when a tool failed, did the agent recover gracefully?

**Evaluation methods:**
1. **Human trajectory rating:** annotators rate each Thought/Action/Observation as good/neutral/bad
2. **LLM-as-judge:** "Rate this agent trajectory on efficiency and quality 1-5"
3. **Environment-based:** simulated environment with ground truth (best for reproducibility)

**Benchmarks:** SWE-bench (software engineering), WebArena (web navigation), AgentBench (diverse tasks)


### Q17. What is the difference between offline and online evaluation for AI systems?
**A:**
**Offline evaluation:**
- Run evaluation on a static test set before deployment
- Controlled, reproducible, fast
- Tests: golden dataset comparisons, automated metrics, LLM-as-judge
- Limitation: test set may not represent real user distribution; can overfit

**Online evaluation (production monitoring):**
- Measure quality on real user traffic in production
- Captures real distribution, real edge cases, real user satisfaction
- Methods: A/B testing, user feedback (thumbs up/down), shadow deployment comparison
- Limitation: expensive if quality is bad (affects real users), harder to attribute cause

**Best practice:** Use offline evaluation to gate deployments; use online evaluation to continuously validate and discover new failures.

**Shadow deployment (hybrid):** run new system in parallel with production, logging outputs without serving them to users. Then compare offline with no user impact.


### Q18. How do you measure factual consistency in LLM outputs?
**A:** Factual consistency = every factual claim in the output is supported by the provided source (for RAG) or is verifiably true (for general knowledge).

**Methods:**

1. **NLI (Natural Language Inference):** For each claim in the output, check if it's entailed by the source context. Models: TRUE (Google), minicheck.
   ```
   NLI(premise=source_context, hypothesis=claim) → entailment | neutral | contradiction
   ```

2. **QA-based consistency:** Generate questions from the output, then answer them from the source; if answers match, the output is consistent.

3. **LLM-as-judge:**
   ```
   "For each factual statement in the response below, classify as:
   [SUPPORTED] - directly from context | [UNSUPPORTED] - not in context | [CONTRADICTED]"
   ```

4. **Factual QA benchmarks:** TruthfulQA, FEVER — test the model's general factual reliability.

🏷️ *Nyaya-Pro: every response includes cited BNS/BNSS sections. I verify programmatically that cited section numbers exist in the knowledge base.*


### Q19. How do you evaluate multi-turn conversation quality?
**A:**
**Dimensions to evaluate:**
1. **Coherence:** Does each response follow logically from the conversation history?
2. **Consistency:** Does the model contradict itself across turns?
3. **Context retention:** Does it correctly remember facts stated earlier in the conversation?
4. **Resolution rate:** Is the user's goal achieved by end of conversation?
5. **Turn efficiency:** How many turns did it take to resolve the query?

**Methods:**
1. **Scripted conversation evaluation:** Pre-write multi-turn conversations with expected outcomes; run the model through them
2. **User simulation:** Use an LLM as a simulated user; evaluate how the agent handles the conversation
3. **Human evaluation:** Have evaluators rate conversation transcripts holistically
4. **Automated probes:** At specific turns in the conversation, ask about something said 5 turns ago; check if the model remembers correctly

**Metrics:** Dialogue Act Accuracy, Resolution Rate@k (resolved within k turns), User Satisfaction Rating.


### Q20. What is the role of golden datasets in AI evaluation?
**A:** A golden dataset = a curated set of (input, expected_output) pairs that serve as the ground truth for evaluation. It's the most valuable evaluation asset you have.

**Characteristics of good golden datasets:**
- **Representative:** covers the full distribution of real user queries
- **Diverse:** edge cases, different query types, different difficulty levels
- **Expert-verified:** answers reviewed by domain experts, not just annotators
- **Versioned:** tracked in git; changes require review
- **Sized appropriately:** 100-500 examples for evaluation; thousands for training

**How to build one:**
1. Collect real user queries from production (anonymised)
2. Have domain experts write verified answers
3. Include cases the system previously failed on
4. Balance difficulty levels

**How to use:**
- Run every model/prompt change against the golden set before deploying
- Track scores over time (quality trend)
- Use for A/B comparison of model variants


### Q21. How do you implement continuous evaluation for production AI systems?
**A:**
**Architecture:**
```
Production Traffic → Sample N% → Shadow Evaluation Pipeline
                                → Automated metrics (faithfulness, format compliance)
                                → LLM-as-judge (async, nightly)
                                → Dashboard update
                                → Alert if metric drops > threshold
```

**What to run continuously:**
1. **Real-time:** format compliance check (JSON valid?), safety filter pass rate, latency
2. **Near-real-time (every hour):** sample 100 requests → faithfulness check → dashboard
3. **Daily:** LLM-as-judge on daily sample; compare vs baseline
4. **Weekly:** full golden dataset eval; human spot-check of 20 random outputs
5. **On each deployment:** run full eval suite before and after; compare; alert if regression

**Tools:** LangSmith for continuous tracing, Prometheus for metrics, custom cron jobs for scheduled evals.

🏷️ *Nyaya-Pro: LangSmith traces every request; weekly manual review of 10 sampled conversations; monthly RAGAS eval on golden dataset.*


### Q22. How do you evaluate bias in AI model outputs?
**A:**
**Types of bias to test:**
- **Demographic bias:** do outputs differ based on race, gender, age, religion, nationality?
- **Representation bias:** does the model have less knowledge about certain groups?
- **Sentiment bias:** consistently more negative/positive about certain groups?
- **Stereotype reinforcement:** does the model perpetuate stereotypes?

**Evaluation methods:**
1. **Counterfactual testing:** identical prompts with only demographic term changed ("he applied for a loan" vs "she applied for a loan"). Compare outputs.
2. **Stereoset/WinoBias:** standard benchmarks for measuring stereotype bias in LLMs
3. **Sentiment analysis across groups:** generate 100 descriptions of different demographic groups; measure sentiment polarity distribution
4. **Representative benchmark coverage:** measure performance gap between groups on the same tasks (e.g., does it answer questions about Country A better than Country B?)

**Mitigation:** Diverse training data, RLHF with diverse annotators, adversarial debiasing, output filtering.


### Q23. How do you compare two models or prompts in a statistically rigorous way?
**A:**
1. **Define the metric** — accuracy, win rate (A vs B), ROUGE score, etc.
2. **Gather sufficient samples** — use power analysis to determine needed sample size; typically 100-500 examples for 80% power
3. **Bootstrap confidence intervals:**
```python
import numpy as np
scores_a = [...]  # scores for model A on N examples
scores_b = [...]  # scores for model B on same N examples
diffs = [a - b for a, b in zip(scores_a, scores_b)]
# Bootstrap CI
boot_means = [np.mean(np.random.choice(diffs, len(diffs))) for _ in range(10000)]
ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])
```
4. **Paired t-test** — for continuous metrics (same examples evaluated by both): `scipy.stats.ttest_rel(scores_a, scores_b)`
5. **Interpret:** if CI doesn't include 0, the difference is statistically significant. Report effect size, not just p-value.
6. **Be wary of multiple comparisons:** if you test 20 variants and 1 is "significantly" better, it may be chance. Apply Bonferroni correction.


### Q24. How do you evaluate the robustness of an LLM application across input variations?
**A:** Robustness = consistent performance even when the input is rephrased, misspelled, or slightly modified.

**Test types:**
1. **Paraphrase robustness:** Rephrase the same query 5 different ways; expect the same answer. Measure variance in output.
2. **Typo/noise robustness:** Add realistic typos ("waht is bail provisoin") → should still answer correctly
3. **Adversarial suffix:** Add irrelevant text to the query — does it distract the model?
4. **Length variation:** Very short vs very long versions of the same query → consistent output?
5. **Format variation:** Question as a statement vs interrogative vs imperative
6. **Language mix:** Mix Hindi/English in a query (common for Indian users) → handle gracefully?

**Metric:** Variance in output quality across N paraphrases of the same intent. Lower variance = more robust.

🏷️ *Nyaya-Pro test: "bail under BNS for murder" | "what are bail provisions for murder according to BNS" | "can someone arrested for murder get bail" → all should cite BNS Section 437.*


### Q25. Key differences between evaluating traditional ML vs LLM applications.
**A:**
| | Traditional ML | LLM Applications |
|--|---------------|-----------------|
| **Output type** | Fixed (class, score, bounding box) | Variable-length text |
| **Evaluation metric** | Accuracy, F1, AUC — clear formulas | No universal metric; depends on task |
| **Reference answer** | Always exists | Sometimes doesn't exist |
| **Evaluation cost** | Cheap (deterministic) | Expensive (LLM-as-judge or human) |
| **Stochasticity** | Deterministic (usually) | Stochastic — same input → different outputs |
| **Drift detection** | Model drift, data drift | Prompt sensitivity, model update drift |
| **Test set size** | 10K+ often needed | 100-500 can be sufficient |
| **Failure modes** | Distribution shift, concept drift | Hallucination, prompt injection, jailbreak |
| **Speed of iteration** | Retrain: days/weeks | Prompt change: seconds |

LLM evaluation is harder because outputs are open-ended and "correctness" is often subjective. Invest in human evaluation even if it's expensive — it's your ground truth.


### Q26. How do you set up an evaluation framework from scratch for a new LLM application?
**A:**
**Step-by-step:**

1. **Define success** (Day 1): What does "good" look like? Pick 3-5 concrete metrics.

2. **Build golden dataset** (Week 1): Collect 100 real (or realistic) user queries. Have domain experts write ideal answers. Store in JSON/CSV, version in git.

3. **Implement automated evaluation** (Week 1): Choose metrics appropriate for your task. Write evaluation script. Run it. Get a baseline score.

4. **LLM-as-judge** (Week 2): For quality dimensions hard to measure automatically, set up LLM-judge with a rubric. Validate against human ratings on 20 examples.

5. **CI integration** (Week 2): Run eval automatically on every pull request. Fail if score drops > 2%.

6. **Production monitoring** (Week 3+): Sample 1% of production traffic → run automated eval daily → dashboard.

7. **Human review cadence** (Ongoing): Weekly manual review of 10-20 sampled outputs. Monthly deep dive.

🏷️ *I set up this framework for Nyaya-Pro in the first 2 weeks. It's caught 3 prompt regressions before they reached production.*


### Q27-29: Evaluation Scenarios

### Q27. Your model passes one fairness metric but fails another. How do you handle conflicting audit results?
**A:** Different fairness metrics (demographic parity, equal opportunity, predictive equality) are mathematically incompatible — improving one often worsens another (Chouldechova's impossibility theorem).

Steps:
1. **Understand the conflict:** Which metric is failing? Who is harmed? What is the magnitude?
2. **Prioritise by harm:** Focus on minimising the most harmful disparities first.
3. **Stakeholder decision:** This is a policy choice, not a technical one. Get legal, ethics, and business leadership involved.
4. **Document the trade-off:** Be transparent about which metrics you're optimising and which you're accepting trade-offs on.
5. **Monitor both:** Even if you can't optimise both simultaneously, monitor both continuously.

---

### Q28. Your model was fair at deployment, but became biased 6 months later. How do you monitor continuously?
**A:** Data drift changes the input distribution → fairness metrics drift.
1. Monitor demographic distributions in inputs and outputs continuously
2. Scheduled fairness audits (monthly) on a representative sample
3. Alert when fairness metric drops > threshold
4. Bias drift detectors: Evidently AI, Whylogs, Fiddler AI

---

### Q29. An external auditor can't reproduce your model's results. How do you ensure audit reproducibility?
**A:**
1. **Pin everything:** model version, prompt version, temperature (use 0 for deterministic), random seeds
2. **Log all inputs and outputs:** immutable, append-only audit log with timestamp + version info
3. **Version-controlled evaluation code and test sets**
4. **Infrastructure-as-code:** reproducible deployment environment (Docker, Terraform)
5. **Provide auditor access** to your evaluation pipeline and frozen test set
